# Paper §3.3: Complex geometry, flow past the island of Madeira

This notebook reproduces the vortex-street figure of Section 3.3 of the paper *GeLB: A Domain-Specific Language and Compiler for Lattice Boltzmann Simulations* (Chirila & Lohmann, submitted to SoftwareX, 2026), with the GeLB programs the paper used ([`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper), release v1.0.0).

Two open-boundary flows with the same numerics: D2Q9 BGK, a Zou–He velocity inlet along the top row, a zero-gradient outlet along the bottom row, periodic sides and half-way bounce-back on the obstacle. Flow runs from top to bottom.

- **(a) Cylinder**, the validation anchor: diameter 48 cells, Re = 150, 200 000 steps on a 512 × 1536 grid.
- **(b) Madeira**: the island's coastline from ETOPO 2022 bathymetry, Re = 300, inflow from the north-north-east, 250 000 steps on a 768 × 1536 grid. The island keeps its map orientation, so the street trails to the south-south-west, as in satellite images of the island's atmospheric wake. As the paper says, this is not meant as a realistic atmospheric simulation: it shows the same lattice Boltzmann program carried over from an idealized obstacle to a real coastline with minimal changes.

**Before running**: switch the runtime to a GPU via *Runtime → Change runtime type → T4 GPU*. The two runs are about 500 billion lattice-node updates, so they use GeLB's CUDA backend. Expected runtime on a T4: **15–20 minutes**, most of it the Madeira run (8 minutes on a laptop RTX 4080).

## 1. Confirm a GPU and the CUDA compiler

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi', '-L'], text=True))
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError('No GPU detected. Switch the runtime via '
                       'Runtime → Change runtime type → T4 GPU, then re-run.')
!nvcc --version | tail -2

The CUDA C++ backend writes its gauge output with the NetCDF-C library:

In [ ]:
!apt-get install -y -qq libnetcdf-dev > /dev/null

## 2. Install GeLB

GeLB 1.0.0, the release described in the paper, installs from its tag on GitHub (about 30 s). The CPU runs use the Fortran backend (`-b cpu_fortran_serial`): a pip installation has no Highway library, which the default `cpu_highway` backend needs (see the [README](https://github.com/dchirila/gelb#installation)). The backends compute the same results; the paper's timings used `cpu_highway`.

In [ ]:
!pip install -q "git+https://github.com/dchirila/gelb.git@v1.0.0"

# gelb-run starts the compiler and its progress monitor through a Python
# interpreter that can import gelb: name this notebook's own.
import os, sys
os.environ["GELB_PYTHON"] = sys.executable

!python -c "import importlib.metadata as m; print('gelb', m.version('gelb'))"

GeLB source is shown with syntax highlighting by the Pygments lexer that the GeLB manuals use:

In [ ]:
!curl -sSfL -o gelb_lexer.py https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/docs/_ext/gelb_lexer.py

from gelb_lexer import GelbLexer
from pygments import highlight
from pygments.formatters import HtmlFormatter
from IPython.display import HTML, display

def show_gelb(path, max_height=None):
    """Display a .gelb file with syntax highlighting and line numbers."""
    with open(path) as f:
        html = highlight(f.read(), GelbLexer(),
                         HtmlFormatter(noclasses=True, style="friendly", linenos="inline",
                                       prestyles="font-family: monospace; font-size: 13px; line-height: 125%;"))
    box = f"max-height: {max_height}px; overflow: auto;" if max_height else ""
    display(HTML(f'<div style="{box}">{html}</div>'))

## 3. Get the paper's programs

In [ ]:
BASE = "https://raw.githubusercontent.com/dchirila/gelb/v1.0.0/examples/softwarex-2026-paper"
for name in ["flow_past_cylinder_2d_bgk_zouhe.gelb", "flow_past_madeira_2d_bgk_zouhe.gelb"]:
    !curl -sSfL -o {name} {BASE}/{name}
!ls -l *.gelb

The Madeira program, as listed (elided) in the paper's appendix. Its `geometry` block paints the coastline cell by cell; it was generated from the bathymetry by [`gen_flow_past_madeira.py`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper/gen_flow_past_madeira.py).

In [ ]:
show_gelb('flow_past_madeira_2d_bgk_zouhe.gelb', max_height=500)

## 4. Run both flows on the GPU

`gelb-run` compiles each program with `gelbc --backend gpu_cuda_cpp`, builds it with `nvcc` for the T4 (`sm_75`), and runs it; the gauge writes the velocity field every 10 000 steps.

In [ ]:
import os
os.environ['GELB_NVCC_ARCH'] = '-arch=sm_75'   # T4 = compute capability 7.5
!gelb-run -b gpu_cuda_cpp -n 200000 -w 10000 flow_past_cylinder_2d_bgk_zouhe.gelb

In [ ]:
!gelb-run -b gpu_cuda_cpp -n 250000 -w 10000 flow_past_madeira_2d_bgk_zouhe.gelb

## 5. The paper's figure

The panels show the cross-stream velocity, the component perpendicular to the inflow: it is zero in the free stream, so the alternating vortices of the street stand out. Land and solid cells are gray. Compare with the paper's figure.

In [ ]:
import netCDF4
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The paper's palette (Okabe-Ito): blue = read / cold, vermillion = write / warm.
BLUE, VERMILLION, GREEN, GRAY, INK = "#0072B2", "#D55E00", "#009E73", "#878787", "#333333"
DIVERGING = LinearSegmentedColormap.from_list("gelb_div", [BLUE, "#f4f3f1", VERMILLION])

def last_record(nc, *names):
    """The last gauge record of each variable, with non-fluid cells as NaN."""
    with netCDF4.Dataset(nc) as ds:
        return [np.ma.filled(ds.variables[n][-1], np.nan) for n in names]

In [ ]:
import math

def cross_stream(nc, rec, azimuth_deg, centre_frac, diameter):
    """Velocity perpendicular to the inflow for one record; colour limit from the wake 2-12 diameters downstream."""
    with netCDF4.Dataset(nc) as ds:
        ux = np.ma.masked_invalid(np.ma.filled(ds.variables["ux"][rec], np.nan))
        uy = np.ma.masked_invalid(np.ma.filled(ds.variables["uy"][rec], np.nan))
        t = int(ds.variables["time"][rec])
    a = math.radians(azimuth_deg)
    ut = ux * math.cos(a) - uy * math.sin(a)
    ut = ut - np.ma.median(ut)
    cy = int(centre_frac * ut.shape[0])
    band = np.abs(ut[max(0, cy - 12 * diameter):cy - 2 * diameter, :].filled(np.nan))
    u_up = float(np.nanmean(np.hypot(ux[cy + 2 * diameter:cy + 3 * diameter, :],
                                     uy[cy + 2 * diameter:cy + 3 * diameter, :])))
    return ut, 0.85 * np.nanmax(band), t, u_up

cmap = DIVERGING.copy(); cmap.set_bad("#7a7a72")
f_a, lim_a, t_a, u_a = cross_stream("flow_past_cylinder_2d_bgk_zouhe.nc", -1, 0.0, 0.75, 48)
f_b, lim_b, t_b, u_b = cross_stream("flow_past_madeira_2d_bgk_zouhe.nc", -1, 22.5, 0.72, 80)
print(f"cylinder: step {t_a}, upstream speed {u_a:.4f}, Re = {u_a * 48 / 0.016:.0f}")
print(f"Madeira:  step {t_b}, upstream speed {u_b:.4f}, Re = {u_b * 80 * 75:.0f}")

fig, axes = plt.subplots(1, 2, figsize=(7, 8),
                         gridspec_kw={"width_ratios": [f_a.shape[1], f_b.shape[1]]})
for ax, f, lim, tag in ((axes[0], f_a, lim_a, "(a) cylinder"), (axes[1], f_b, lim_b, "(b) Madeira")):
    ny, nx = f.shape
    ax.imshow(f, origin="lower", cmap=cmap, vmin=-lim, vmax=lim,
              extent=(0, nx, 0, ny), interpolation="bilinear")
    ax.text(0.03, 0.988, tag, transform=ax.transAxes, va="top", ha="left", color=INK,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5))
    ax.set_xticks([]); ax.set_yticks([])
axes[1].annotate("N", xy=(0.93, 0.925), xytext=(0.93, 0.985), xycoords="axes fraction",
                 textcoords="axes fraction", ha="center", va="top", color=INK,
                 arrowprops=dict(arrowstyle="<-", color=INK, lw=0.9))
plt.tight_layout(); plt.show()

## 6. The street develops

The cylinder's wake at four of the gauge records: the street forms behind the cylinder and travels downstream.

In [ ]:
with netCDF4.Dataset("flow_past_cylinder_2d_bgk_zouhe.nc") as ds:
    n_rec = ds.variables["time"].shape[0]
records = [2, n_rec // 4, n_rec // 2, n_rec - 1]
fig, axes = plt.subplots(1, len(records), figsize=(10, 7))
for ax, rec in zip(axes, records):
    f, lim, t, _ = cross_stream("flow_past_cylinder_2d_bgk_zouhe.nc", rec, 0.0, 0.75, 48)
    ax.imshow(f, origin="lower", cmap=cmap, vmin=-lim_a, vmax=lim_a, interpolation="bilinear")
    ax.set_title(f"step {t}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Where to go from here

- The programs and the commands that produced the paper's figures: [`examples/softwarex-2026-paper/`](https://github.com/dchirila/gelb/tree/v1.0.0/examples/softwarex-2026-paper).
- The [GeLB User Tutorial](https://github.com/dchirila/gelb/blob/v1.0.0/docs/pdf/GeLB_User_Tutorial.pdf) (PDF) and the [repository](https://github.com/dchirila/gelb).
- Release 1.0.0 is archived as [doi:10.5281/zenodo.23592922](https://doi.org/10.5281/zenodo.23592922); the [README](https://github.com/dchirila/gelb#using-gelb-in-your-work) says how to cite GeLB.